# Edit Actions Analysis

Load `edit-actions/*/analysis.db`, pull paired versions, render side-by-side diffs, and inspect extracted edit actions.


In [1]:
from pathlib import Path
import sys
import sqlite3
import gzip
import shutil
import tempfile
import importlib
import pandas as pd
from IPython.display import display, Markdown

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "util.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import util as util_module
importlib.reload(util_module)
print(f"Loaded util from: {Path(util_module.__file__).resolve()}")

highlight_pair = util_module.highlight_pair
format_article_html = util_module.format_article_html



Loaded util from: /Users/spangher/Projects/stanford-research/matt-debutts/notebooks/util.py


In [2]:
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
EDIT_ACTIONS_ROOT = ROOT / "edit-actions"
SOURCE_DB_ROOT = ROOT / "article-versions"

analysis_dbs = sorted(EDIT_ACTIONS_ROOT.glob("*/analysis.db"))
analysis_dbs

[PosixPath('/Users/spangher/Projects/stanford-research/matt-debutts/edit-actions/newssniffer-guardian/analysis.db'),
 PosixPath('/Users/spangher/Projects/stanford-research/matt-debutts/edit-actions/newssniffer-nytimes/analysis.db'),
 PosixPath('/Users/spangher/Projects/stanford-research/matt-debutts/edit-actions/newssniffer-washpo/analysis.db')]

In [3]:
sql = """
SELECT
  news_org,
  article_id,
  from_version_id,
  to_version_id,
  from_version_num,
  to_version_num,
  pair_policy,
  editor_request,
  writer_action,
  content_added,
  content_removed,
  content_changed
FROM pair_edit_actions
"""

frames = []
for db_path in analysis_dbs:
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql(sql, conn)
    if not df.empty:
        df["analysis_db"] = db_path
        frames.append(df)

pairs_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
pairs_df.head()


,news_org,article_id,from_version_id,to_version_id,from_version_num,to_version_num,pair_policy,editor_request,writer_action,content_added,content_removed,content_changed,analysis_db
0,newssniffer-guardian,487220,487220-0,487220-15,0,15,pair_list_generated,None,None,None,None,None,/Users/spangher/Projects/stanford-research/mat...
1,newssniffer-guardian,487220,487220-0,487220-1,0,1,pair_list_generated,Provide more details about the candidates' sch...,Researched and compiled the candidates' itiner...,Newt Gingrich kicked the day off with an 8.30a...,None,None,/Users/spangher/Projects/stanford-research/mat...
2,newssniffer-guardian,487220,487220-0,487220-1,0,1,pair_list_generated,Remove the quote about violating vows from the...,Edited the article to remove the sensitive quote,None,"""If you are going to violate those vows – let'...",None,/Users/spangher/Projects/stanford-research/mat...
3,newssniffer-guardian,487220,487220-0,487220-1,0,1,pair_list_generated,Update the timestamp to reflect the current time,Updated the timestamp,None,9.13am:,9.32am:,/Users/spangher/Projects/stanford-research/mat...
4,newssniffer-guardian,487220,487220-1,487220-2,1,2,pair_list_generated,Add more information about Mitt Romney's perso...,Conducted research and added a brief story abo...,10.11am: Last night Mitt Romney really let his...,None,None,/Users/spangher/Projects/stanford-research/mat...


In [4]:
import atexit

_unzipped_cache = {}
_summary_lookup_cache = {}
UNZIP_SESSION_DIR = Path('../unzipped-article-versions')
print(f"Temporary unzip dir: {UNZIP_SESSION_DIR}")


def _source_db_path(news_org: str) -> Path:
    stem = news_org
    if stem.startswith("newssniffer-"):
        stem = stem[len("newssniffer-"):]
    direct_db = SOURCE_DB_ROOT / f"{news_org}.db"
    direct_gz = SOURCE_DB_ROOT / f"{news_org}.db.gz"
    alt_db = SOURCE_DB_ROOT / f"newssniffer-{stem}.db"
    alt_gz = SOURCE_DB_ROOT / f"newssniffer-{stem}.db.gz"
    ap_db = SOURCE_DB_ROOT / "ap.db"

    for candidate in (direct_db, direct_gz, alt_db, alt_gz, ap_db):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No source DB found for {news_org}")


def _materialize_sqlite(db_path: Path) -> Path:
    if db_path.suffix != ".gz":
        return db_path
    if db_path in _unzipped_cache:
        return _unzipped_cache[db_path]

    # Temporary unzipped copy for this notebook session only.
    tmp_path = UNZIP_SESSION_DIR / f"{db_path.stem}.db"
    if not tmp_path.exists() or tmp_path.stat().st_mtime < db_path.stat().st_mtime:
        with gzip.open(db_path, "rb") as src, open(tmp_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    _unzipped_cache[db_path] = tmp_path
    return tmp_path


def _build_summary_lookup(news_org: str):
    db = _materialize_sqlite(_source_db_path(news_org))
    if db in _summary_lookup_cache:
        return _summary_lookup_cache[db]

    with sqlite3.connect(db) as conn:
        df = pd.read_sql_query(
            "SELECT entry_id, version, summary FROM entryversion",
            conn,
        )

    lookup = {
        (int(row.entry_id), int(row.version)): (row.summary or "")
        for row in df.itertuples(index=False)
    }
    _summary_lookup_cache[db] = lookup
    print(f"Loaded {len(lookup):,} summaries into memory from {db.name}")
    return lookup


def cleanup_unzipped_cache() -> None:
    """Delete temporary unzipped DBs created for this session."""
    _summary_lookup_cache.clear()
    for path in list(_unzipped_cache.values()):
        try:
            path.unlink(missing_ok=True)
        except OSError:
            pass
    _unzipped_cache.clear()
    try:
        UNZIP_SESSION_DIR.rmdir()
    except OSError:
        pass


def load_version_summary(news_org: str, entry_id: str, version_id: str) -> str:
    lookup = _build_summary_lookup(news_org)
    try:
        key = (int(entry_id), int(version_id))
    except (TypeError, ValueError):
        return ""
    return lookup.get(key, "")


# Clean up temp unzipped files automatically when kernel exits.
atexit.register(cleanup_unzipped_cache)



Temporary unzip dir: ../unzipped-article-versions


<function __main__.cleanup_unzipped_cache() -> None>

In [10]:
# Sample and inspect grouped by version pair
import difflib

N = 5
RANDOM_STATE = 12
PAIR_MODE = "adjacent"  # options: "adjacent", "first_last"

# Filters for advisor examples
MAX_WORDS_PER_VERSION = 600
TARGET_CHANGE = 0.20
CHANGE_TOLERANCE = 0.05  # keep pairs in [TARGET_CHANGE +/- CHANGE_TOLERANCE]

pair_key_cols = [
    "news_org",
    "article_id",
    "from_version_id",
    "to_version_id",
    "from_version_num",
    "to_version_num",
    "pair_policy",
]

summary_cache = {}

def _cached_summary(news_org, article_id, version_num):
    key = (news_org, article_id, version_num)
    if key not in summary_cache:
        summary_cache[key] = load_version_summary(news_org, article_id, version_num) or ""
    return summary_cache[key]


def _change_ratio(left_text: str, right_text: str) -> float:
    return 1.0 - difflib.SequenceMatcher(None, left_text, right_text).ratio()


if pairs_df.empty:
    sample_pairs_df = pd.DataFrame(columns=pair_key_cols)
else:
    pair_rows = pairs_df[pair_key_cols].drop_duplicates().copy()

    if PAIR_MODE == "adjacent":
        pair_rows = pair_rows[pair_rows["to_version_num"] == pair_rows["from_version_num"] + 1]
    elif PAIR_MODE == "first_last":
        bounds = (
            pair_rows.groupby(["news_org", "article_id"], as_index=False)
            .agg(first_version=("from_version_num", "min"), last_version=("to_version_num", "max"))
        )
        pair_rows = pair_rows.merge(bounds, on=["news_org", "article_id"], how="left")
        pair_rows = pair_rows[
            (pair_rows["from_version_num"] == pair_rows["first_version"])
            & (pair_rows["to_version_num"] == pair_rows["last_version"])
        ].drop(columns=["first_version", "last_version"])
    else:
        raise ValueError('PAIR_MODE must be "adjacent" or "first_last"')

In [11]:
from tqdm.auto import tqdm

In [12]:
filtered_rows = []
for _, pair in tqdm(pair_rows.iterrows(), total=len(pair_rows)):
    summary_t1 = _cached_summary(pair["news_org"], pair["article_id"], pair["from_version_num"])
    summary_t2 = _cached_summary(pair["news_org"], pair["article_id"], pair["to_version_num"])

    words_t1 = len(summary_t1.split())
    words_t2 = len(summary_t2.split())
    if max(words_t1, words_t2) > MAX_WORDS_PER_VERSION:
        continue

    change_ratio = _change_ratio(summary_t1, summary_t2)
    if abs(change_ratio - TARGET_CHANGE) > CHANGE_TOLERANCE:
        continue

    row = pair.to_dict()
    row["change_ratio"] = change_ratio
    row["words_t1"] = words_t1
    row["words_t2"] = words_t2
    filtered_rows.append(row)

pair_rows = pd.DataFrame(filtered_rows)

sample_pairs_df = (
    pair_rows.sample(min(N, len(pair_rows)), random_state=RANDOM_STATE)
    if not pair_rows.empty
    else pair_rows
)

sample_pairs_df

  0%|          | 0/1035 [00:00<?, ?it/s]

,news_org,article_id,from_version_id,to_version_id,from_version_num,to_version_num,pair_policy,change_ratio,words_t1,words_t2
8,newssniffer-nytimes,957476,957476-0,957476-1,0,1,pair_list_generated,0.178976,398,554
7,newssniffer-nytimes,842513,842513-0,842513-1,0,1,pair_list_generated,0.179547,299,371
0,newssniffer-guardian,494042,494042-0,494042-1,0,1,pair_list_generated,0.220593,400,547
4,newssniffer-nytimes,695511,695511-1,695511-2,1,2,pair_list_generated,0.189920,550,520
3,newssniffer-guardian,515724,515724-0,515724-1,0,1,pair_list_generated,0.219442,152,175


In [13]:
for _, pair in sample_pairs_df.iterrows():
    pair_mask = (
        (pairs_df["news_org"] == pair["news_org"])
        & (pairs_df["article_id"] == pair["article_id"])
        & (pairs_df["from_version_num"] == pair["from_version_num"])
        & (pairs_df["to_version_num"] == pair["to_version_num"])
    )
    actions_df = pairs_df.loc[
        pair_mask,
        ["editor_request", "writer_action", "content_added", "content_removed", "content_changed"],
    ].reset_index(drop=True)

    summary_t1 = _cached_summary(pair["news_org"], pair["article_id"], pair["from_version_num"])
    summary_t2 = _cached_summary(pair["news_org"], pair["article_id"], pair["to_version_num"])

    left_html, right_html = highlight_pair(summary_t1, summary_t2)

    display(
        format_article_html(
            pair["article_id"],
            pair["from_version_num"],
            pair["to_version_num"],
            left_html,
            right_html,
        )
    )

    display(
        Markdown(
            f"**news_org**: `{pair['news_org']}` | **pair_policy**: `{pair['pair_policy']}` | "
            f"**change_ratio**: `{pair['change_ratio']:.3f}` | "
            f"**words**: `{int(pair['words_t1'])}` -> `{int(pair['words_t2'])}` | "
            f"**edit-actions found**: `{len(actions_df)}`"
        )
    )

    for action_idx, action_row in actions_df.iterrows():
        display(Markdown(f"### Edit action {action_idx + 1}"))
        display(Markdown(f"**Editor request**\n\n{action_row['editor_request'] or ''}"))
        display(Markdown(f"**Writer action**\n\n{action_row['writer_action'] or ''}"))
        display(Markdown(f"**Content added**\n\n{action_row['content_added'] or ''}"))
        display(Markdown(f"**Content removed**\n\n{action_row['content_removed'] or ''}"))
        display(Markdown(f"**Content changed**\n\n{action_row['content_changed'] or ''}"))

    display(Markdown("---"))

**news_org**: `newssniffer-nytimes` | **pair_policy**: `pair_list_generated` | **change_ratio**: `0.179` | **words**: `398` -> `554` | **edit-actions found**: `3`

### Edit action 1

**Editor request**

Provide more context on the international community's response to North Korea's actions

**Writer action**

Researched recent statements from world leaders and added information about Secretary of State John Kerry's visit to South Korea

**Content added**

On Monday, Secretary of State John Kerry, during his visit to South Korea, lambasted the North Korean leader over the reported atrocities and called for greater international pressure. Alas, there are not many sanctions that would have an effect on a regime already isolated and immune to the suffering of its citizens. Moreover, Mr. Kim has shown no interest in resuming the six-nation talks, suspended in 2008, on putting an end to North Korea’s nuclear weapons program. Indeed, Pyongyang is said to be working on an intercontinental missile capable of carrying a nuclear warhead. Despite these obstacles, finding more ways to pressure Mr. Kim, perhaps involving China, is a must.

...

**Writer action**

Made minor edits to improve sentence flow and clarity

**Content added**



**Content removed**



**Content changed**

A spokesman from Nato headquarters in the country said it was aware of the attack

---

In [7]:
import pyperclip
pyperclip.copy(right_html)

In [69]:
sample_df

,news_org,article_id,from_version_id,to_version_id,from_version_num,to_version_num,pair_policy,editor_request,writer_action,content_added,content_removed,content_changed,analysis_db
1683,newssniffer-guardian,518987,518987-0,518987-4,0,4,pair_list_generated,Correct minor errors in proper nouns,Corrected the spelling of 'Hilary Clinton' to ...,None,None,'Hilary Clinton' changed to 'Hillary Clinton',/Users/spangher/Projects/stanford-research/mat...
1719,newssniffer-guardian,518776,518776-1,518776-2,1,2,pair_list_generated,Change 'had insisted that his inspectors shoul...,Edited the sentence to remove the word 'that',None,that,"For his part, Herman Nackaerts, the chief IAEA...",/Users/spangher/Projects/stanford-research/mat...
507,newssniffer-guardian,493832,493832-8,493832-9,8,9,pair_list_generated,Remove the section about the first anniversary...,Deleted the introductory paragraph about Libya,None,Friday marks the first anniversary of the upri...,None,/Users/spangher/Projects/stanford-research/mat...
868,newssniffer-guardian,503944,503944-1,503944-2,1,2,pair_list_generated,Include information about the UK motor industry,Added information about the UK motor industry,Better news from the UK motor industry though ...,None,None,/Users/spangher/Projects/stanford-research/mat...
593,newssniffer-guardian,495939,495939-0,495939-10,0,10,pair_list_generated,Include more information about the Centre for ...,Added data and quotes from the Centre for Resp...,"The Centre for Responsive Politics, otherwise ...",None,None,/Users/spangher/Projects/stanford-research/mat...


In [44]:
left_html, right_html = highlight_pair(summary_t1, summary_t2)

In [57]:
db = _materialize_sqlite(_source_db_path(row['news_org']))

In [58]:
db

PosixPath('../unzipped-article-versions/newssniffer-guardian.db.db')

In [60]:
con = sqlite3.connect(db)

In [65]:
q = f"SELECT summary FROM entryversion WHERE entry_id = {row['article_id']} AND version = {row['from_version_num']} LIMIT 1"

In [64]:
row

news_org                                         newssniffer-guardian
article_id                                                     495939
from_version_id                                              495939-0
to_version_id                                               495939-10
from_version_num                                                    0
to_version_num                                                     10
pair_policy                                       pair_list_generated
editor_request      Include more information about the Centre for ...
writer_action       Added data and quotes from the Centre for Resp...
content_added       The Centre for Responsive Politics, otherwise ...
content_removed                                                  None
content_changed                                                  None
analysis_db         /Users/spangher/Projects/stanford-research/mat...
Name: 593, dtype: object

In [ ]:
# Optional manual cleanup now (instead of waiting for kernel shutdown):
# cleanup_unzipped_cache()
